# Size scaling

Sweep `n in {4, 6, 8, 10}` at fixed depth `p`, comparing QAOA against the
classical baselines. Honest answer at these sizes: QAOA does not win on
speed — the point is the methodology and quality of approximation.

### SETUP

In [ ]:
import sys, json
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np

from scripts.data      import load_returns
from scripts.portfolio import Portfolio
from scripts.qaoa      import QAOA
from scripts.classical import brute_force, greedy_top_k, simulated_annealing, markowitz_round
from scripts.metrics   import approximation_ratio, prob_optimal

RESULTS_DIR = Path.cwd().parent / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

### TICKER UNIVERSE

Use a stable superset and subsample to size `n`.

In [ ]:
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN',
            'NVDA', 'IBM',  'HON',   'ACN',
            'XOM',  'JPM']
START, END = '2023-01-01', '2025-12-31'

N_VALUES = [4, 6, 8, 10]
P        = 2
K_FRAC   = 0.25  # K = round(K_FRAC * n)
LAM, AP  = 2.0, 0.5

### SWEEP (cached)

In [ ]:
cache = RESULTS_DIR / 'size_scaling.json'

if cache.exists():
    rows = json.loads(cache.read_text())
    print(f'loaded {cache.name}')
else:
    rows = []
    for n in N_VALUES:
        tickers = UNIVERSE[:n]
        K = max(1, round(K_FRAC * n))
        r  = load_returns(tickers, START, END, cache_name=f'scaling_n{n}')
        pf = Portfolio(r.mu, r.Sigma, lam=LAM, A=AP, K=K, tickers=list(r.tickers))

        bf = brute_force(pf)

        qaoa = QAOA(pf, seed=42)
        E0   = qaoa.ground_state_energy()
        qres = qaoa.optimise(p=P, n_restarts=15)

        for solver, sr in [
            ('brute_force',         bf),
            ('greedy_sharpe',       greedy_top_k(pf, score='sharpe')),
            ('markowitz_round',     markowitz_round(pf)),
            ('simulated_annealing', simulated_annealing(pf, n_sweeps=500)),
        ]:
            rows.append({
                'n': n, 'K': K, 'solver': solver,
                'cost': sr.cost, 'runtime_s': sr.runtime,
                'ratio': sr.cost / bf.cost,
            })
        rows.append({
            'n': n, 'K': K, 'solver': f'qaoa_p{P}',
            'cost': float(qres['energy']),
            'runtime_s': None,
            'ratio': approximation_ratio(qres['energy'], E0),
            'p_optimal': prob_optimal(qres['probs'], bf.x),
        })
    cache.write_text(json.dumps(rows, indent=2))
    print(f'saved → {cache.name}')

import pandas as pd
pd.DataFrame(rows)